In [2]:
import numpy as np
import pandas as pd
from minisom import MiniSom
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix

In [3]:
X = np.array([
    [13.3, 18.3, 10.0,  8.0, 11.0,  6.0],
    [18.0, 20.0, 16.0, 10.0, 11.1,  8.9],
    [13.8, 11.3,  7.5,  9.2,  7.5,  5.0],
    [ 8.0,  3.3, 10.0,  6.9,  2.9,  8.6],
    [12.2, 21.7, 10.9,  8.0, 14.3,  7.1],
    [12.0,  5.0, 16.3,  7.4,  3.1, 10.0],
    [11.4,  9.3,  8.1,  8.0,  6.5,  5.7],
    [ 5.7, 10.9,  6.8,  3.9,  7.5,  4.7],
    [15.0, 21.3, 14.0,  9.0, 12.8,  8.4],
    [14.0, 13.0,  9.6, 10.0,  9.3,  6.9],
    [ 9.0, 13.0,  8.0,  6.4,  9.3,  5.7],
    [22.5, 20.0, 22.0, 11.3, 10.0, 11.0],
    [13.0,  5.0,  7.0,  9.3,  3.6,  5.0],
    [ 9.5,  4.5, 16.0,  9.5,  4.5, 16.0],
    [10.0, 25.0,  7.5,  6.7, 16.7,  5.0],
    [12.7,  6.7, 20.0,  6.3,  3.3, 10.0],
    [ 9.7,  9.1,  7.7,  6.2,  5.8,  4.9],
    [ 6.3,  9.5,  5.0,  4.2,  6.3,  3.3],
    [15.0, 22.3, 10.7, 11.3, 16.8,  8.0],
    [11.4, 12.0,  8.0,  9.5, 10.0,  6.7],
])

dea_eff = np.array([
    0.820, 0.942, 0.811, 0.651, 0.946,
    0.819, 0.706, 0.516, 0.963, 0.885,
    0.631, 1.000, 0.817, 1.000, 1.000,
    0.903, 0.547, 0.420, 1.000, 0.842
])

Нормируем признаки:

In [4]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

Обучение сети Кохонена:  
2 нейрона => 2 кластера

In [5]:
som = MiniSom(x=1, y=2, input_len=6, sigma=1.0, learning_rate=0.5,
              neighborhood_function='gaussian', random_seed=42)

som.random_weights_init(X_scaled)
som.train_random(X_scaled, num_iteration=500)

In [6]:
winners = np.array([som.winner(x) for x in X_scaled])

т.к. сеть 1x2, достаточно координаты по оси y

In [7]:
clusters_som = winners[:, 1]

Классы по DEA  
считаем склады "DEA-эффективными", если E >= 0.9

In [8]:
dea_class = (dea_eff >= 0.9).astype(int)

In [10]:
dea_class = (dea_eff >= 0.9).astype(int)

df = pd.DataFrame(X, columns=['3/1','4/1','5/1','3/2','4/2','5/2'])
df.index = np.arange(1, 21)
df['DEA_eff'] = dea_eff
df['DEA_class'] = dea_class
df['SOM_cluster'] = clusters_som

print("Таблица с результатами кластеризации:\n")
print(df)

print("\nМатрица ошибок (DEA_class vs SOM_cluster):")
print(confusion_matrix(dea_class, clusters_som))

accuracy = np.mean(dea_class == clusters_som)
print(f"\nДоля совпавших присвоений кластера и DEA-класса: {accuracy:.3f}")

Таблица с результатами кластеризации:

     3/1   4/1   5/1   3/2   4/2   5/2  DEA_eff  DEA_class  SOM_cluster
1   13.3  18.3  10.0   8.0  11.0   6.0    0.820          0            0
2   18.0  20.0  16.0  10.0  11.1   8.9    0.942          1            0
3   13.8  11.3   7.5   9.2   7.5   5.0    0.811          0            0
4    8.0   3.3  10.0   6.9   2.9   8.6    0.651          0            1
5   12.2  21.7  10.9   8.0  14.3   7.1    0.946          1            0
6   12.0   5.0  16.3   7.4   3.1  10.0    0.819          0            1
7   11.4   9.3   8.1   8.0   6.5   5.7    0.706          0            1
8    5.7  10.9   6.8   3.9   7.5   4.7    0.516          0            1
9   15.0  21.3  14.0   9.0  12.8   8.4    0.963          1            0
10  14.0  13.0   9.6  10.0   9.3   6.9    0.885          0            0
11   9.0  13.0   8.0   6.4   9.3   5.7    0.631          0            1
12  22.5  20.0  22.0  11.3  10.0  11.0    1.000          1            0
13  13.0   5.0   7.0   9.